# MiniDavis — Day 5: prepare, review, then fine-tune

Eric's data/session handoff for Gayatri. The default is an **offline dry run**: no model downloads, training, API calls or test-set metrics. The 200 synthetic examples need assigned human review before training.

We fine-tune a small **DistilBERT interaction classifier**, not the town's hosted Qwen model. Labels are cooperation, conflict, trade and neutral.

Local: open this notebook from the repository. Colab: upload the supplied `minidavis-day5-colab.zip` in the Files panel before running the first cell. After reviewing/freezing data, rebuild the zip from the reviewed repository. Opening a GitHub notebook alone does not bring the data files into Colab.


In [ ]:
from pathlib import Path
import json
import sys
import zipfile

RUN_TRAINING = False
INSTALL_DEPENDENCIES = False

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/MiniDavis')]
ROOT = next((p for p in candidates if (p / 'ml/data/interactions.jsonl').is_file()), None)
if ROOT is None:
    bundle = Path('/content/minidavis-day5-colab.zip')
    if not bundle.is_file():
        raise FileNotFoundError('Upload minidavis-day5-colab.zip in Colab, or run inside the MiniDavis repository.')
    with zipfile.ZipFile(bundle) as archive:
        for name in archive.namelist():
            member = Path(name)
            if member.is_absolute() or '..' in member.parts or member.parts[0] != 'MiniDavis':
                raise ValueError('Unexpected archive member')
        archive.extractall('/content')
    ROOT = Path('/content/MiniDavis')
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / 'ml/data'
print('Project:', ROOT)


## 1. Check the data without training

200 examples, 50/class. Forty families, five versions each. Split whole families into 120 train / 40 validation / 40 test (30/10/10 per class). The check prints counts and hashes only; it does not evaluate the held-out set.


In [ ]:
from ml.interaction_data import validate_dataset

data_report = validate_dataset(DATA_DIR)
print(json.dumps(data_report, indent=2))


## 2. Review before enabling training

Read `ml/data/README.md` for the label policy. Aishani reviews 001–050, Ethan 051–100, Gayatri 101–200. Correct text/labels in the JSONL file and set the assigned rows to approved in `review_queue.csv` only after actual review. Changes to reviewed text need a fresh review.

After corrections, run `python scripts/prepare_interactions.py --refresh-draft`, then `python scripts/prepare_interactions.py --freeze`. Commit the exact reviewed data, CSV and manifest together. Do not move families after model results. Keep Day 6 test metrics untouched.

Training requires a frozen manifest and 200 approved reviews. To train, set both switches in the first cell to True and rerun in a fresh session. No Groq or Hugging Face secret is needed for this public checkpoint. Internet is needed to install packages and download DistilBERT. Select a Colab GPU runtime if available; otherwise training uses CPU batch size 4 and may take longer. This training branch has not been executed in Eric's offline preparation.


In [ ]:
import subprocess

if RUN_TRAINING:
    validate_dataset(DATA_DIR, require_frozen=True)
    if INSTALL_DEPENDENCIES:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'ml/requirements.txt')])
    print('Reviewed/frozen input passed. Restart the runtime if Colab requests it after installation.')
else:
    if INSTALL_DEPENDENCIES:
        raise ValueError('Keep dependency installation off for an offline dry run.')
    print('Dry run: installation and training are disabled. Finish reviews before freezing.')


## 3. Save outside the temporary Colab runtime

Training mounts your Drive and writes to `MiniDavis/Models/day5-v1`. Choose a new output folder for each later run. On a local machine, exports go to ignored `ml/artifacts/day5-v1`. Do not add large model weights to Git. Check that you can reopen the Drive files after the session.


In [ ]:
OUTPUT_DIR = ROOT / 'ml/artifacts/day5-v1'
if RUN_TRAINING and 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/MiniDavis/Models/day5-v1')
print('Output directory:', OUTPUT_DIR)


## 4. Train baseline, fine-tune, validate, export

The helper fits TF-IDF + logistic regression on training text only, then trains `distilbert/distilbert-base-uncased` for three epochs, maximum length 128, learning rate 2e-5, batch size 8 on CUDA or 4 on CPU. It selects the checkpoint by **validation macro-F1**, exports weights/tokenizer/config, and saves baseline and model validation metrics plus exact runtime versions. It does not return test rows to the training routine. Adjust settings using validation only; do not claim that this small synthetic dataset proves real-world performance.


In [ ]:
if RUN_TRAINING:
    from ml.train_classifier import run_training
    training_report = run_training(DATA_DIR, OUTPUT_DIR)
    print(json.dumps(training_report, indent=2))
else:
    print('Skipped: no training, checkpoint download, validation scores or model export were produced.')


## 5. Confirm the export

Gayatri checks that the exported classifier reloads on CPU, then shares the Drive path with Eric/Rylee. Runtime integration and held-out evaluation remain Day 6 work. This artificial sentence is an inference smoke check, not a held-out score or an actual town interaction.


In [ ]:
if RUN_TRAINING:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    exported = OUTPUT_DIR / 'classifier'
    cpu_tokenizer = AutoTokenizer.from_pretrained(exported, local_files_only=True)
    cpu_model = AutoModelForSequenceClassification.from_pretrained(exported, local_files_only=True).to('cpu').eval()
    sample = 'Ethan gives Rylee spare food without asking for payment.'
    inputs = cpu_tokenizer(sample, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probabilities = cpu_model(**inputs).logits.softmax(dim=-1)[0]
    label_id = int(probabilities.argmax())
    print({'fixture': sample, 'prediction': cpu_model.config.id2label[label_id],
           'confidence': float(probabilities[label_id]), 'held_out_score': False})
else:
    print('Dry-run complete. Next: assigned reviews, freeze, then Gayatri runs training in Colab.')


Workflow references: [Hugging Face text classification](https://huggingface.co/docs/transformers/tasks/sequence_classification), [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), [Colab resource availability](https://research.google.com/colaboratory/faq.html). Exact notebook package versions are pinned in `ml/requirements.txt`; record the runtime PyTorch/CUDA versions in every training report.
